In [1]:
%pip install -q pandera pandas pyarrow

import os
import sys
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

VALIDATION_DIR = os.path.join(
    PROJECT_DIR, "src", "validation"
)

sys.path.insert(0, VALIDATION_DIR)

from schema import ratings_schema

print("Validation environment ready!")

Mounted at /content/drive
Validation environment ready!


In [2]:
# invalid test records
ratings_path = os.path.join(
    PROJECT_DIR, "data", "processed", "ratings.parquet"
)

ratings_df = pd.read_parquet(ratings_path)

# Take a small sample without changing the original data
test_ratings = ratings_df.head(10).copy()

# Introduce invalid values deliberately
test_ratings.loc[test_ratings.index[0], "rating"] = 6.0
test_ratings.loc[test_ratings.index[1], "rating"] = -1.0

print("Test dataset created:")
display(test_ratings)

Test dataset created:


,userId,movieId,rating,timestamp
0,1,1,6.0,964982703
1,1,3,-1.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [3]:
#detecting and saving invalid records
import pandera as pa

ERROR_DIR = os.path.join(PROJECT_DIR, "data", "errors")
os.makedirs(ERROR_DIR, exist_ok=True)

try:
    ratings_schema.validate(test_ratings, lazy=True)
    print("Validation passed. No invalid records found.")

except pa.errors.SchemaErrors as error:

    failure_cases = error.failure_cases.copy()

    print("Validation failed as expected!")
    print("\nDetected validation errors:")
    display(failure_cases)

    error_report_path = os.path.join(
        ERROR_DIR, "ratings_validation_errors.csv"
    )

    failure_cases.to_csv(error_report_path, index=False)

    # Identify rows that failed validation
    invalid_indices = pd.to_numeric(
        failure_cases["index"], errors="coerce"
    ).dropna().astype(int).unique()

    invalid_records = test_ratings.loc[
        test_ratings.index.isin(invalid_indices)
    ].copy()

    invalid_records_path = os.path.join(
        ERROR_DIR, "invalid_ratings.csv"
    )

    invalid_records.to_csv(invalid_records_path, index=False)

    print("\nInvalid records detected:", len(invalid_records))
    print("Validation errors saved:", error_report_path)
    print("Invalid records saved:", invalid_records_path)

Validation failed as expected!

Detected validation errors:


,schema_context,column,check,check_number,failure_case,index
0,Column,rating,"in_range(0.5, 5.0)",0,6.0,0
1,Column,rating,"in_range(0.5, 5.0)",0,-1.0,1



Invalid records detected: 2
Validation errors saved: /content/drive/MyDrive/recommendation-pipeline/data/errors/ratings_validation_errors.csv
Invalid records saved: /content/drive/MyDrive/recommendation-pipeline/data/errors/invalid_ratings.csv
